In [13]:
import pandas as pd
import altair as alt

#load CSV
df = pd.read_csv('cleaned_lung_cancer_data.csv')

df.head()

,age,gender,smoking,finger_discoloration,mental_stress,exposure_to_pollution,long_term_illness,energy_level,immune_weakness,breathing_issue,...,oxygen_saturation,chest_tightness,family_history,smoking_family_history,stress_immune,pulmonary_disease,pulmonary_disease_num,age_scaled,energy_level_scaled,oxygen_saturation_scaled
0,68,1,1,1,1,1,0,57.831178,0,0,...,95.977287,1,0,0,0,NO,0,0.682203,0.353770,0.665986
1,81,1,1,0,0,1,1,47.694835,1,1,...,97.184483,0,0,0,0,YES,1,1.505110,-0.927318,1.481163
2,58,1,1,0,0,0,0,59.577435,0,1,...,94.974939,0,0,0,0,NO,0,0.049197,0.574472,-0.010865
3,44,0,1,0,1,1,0,59.785767,0,1,...,95.187900,0,0,0,0,YES,1,-0.837011,0.600802,0.132940
4,72,0,1,1,1,1,1,59.733941,0,1,...,93.503008,0,0,0,0,YES,1,0.935405,0.594252,-1.004809


In [ ]:
rows = []
# include the 3 factors
for factor, col in [
    ('Smoking', 'smoking'),
    ('Family History', 'smoking_family_history'),
    ('Throat Discomfort', 'throat_discomfort'),
]:
    #split patients into two groups (smoking vs non-smoking)
    for val, label in [(1, 'Yes'), (0, 'No')]:
        sub = df[df[col] == val]
        total = len(sub)
        #calculate which portion has the disease
        for disease_val, disease_label in [(1, 'Pulmonary Disease'), (0, 'No Disease')]:
            count = (sub['pulmonary_disease_num'] == disease_val).sum()
            rows.append({
                'Factor': factor,
                'Group': label,
                'Outcome': disease_label,
                'Proportion': count / total,
                'Count': int(count),
                'Total': int(total)
            })

stack_df = pd.DataFrame(rows)

# Dropdown to filter by factor
input_dropdown = alt.binding_select(
    options=[None, 'Smoking', 'Family History', 'Throat Discomfort'],
    labels=['All', 'Smoking', 'Family History', 'Throat Discomfort'],
    name='Factor: '
)
selection = alt.selection_point(fields=['Factor'], bind=input_dropdown)

alt.Chart(stack_df).mark_bar(
    cornerRadiusTopLeft=4,
    cornerRadiusTopRight=4
).encode(
    x = alt.X('Group:N',
              sort=['Yes', 'No'],
              axis=alt.Axis(title=None, labelAngle=0)),
    y = alt.Y('Proportion:Q',
              stack='normalize',
              axis=alt.Axis(title='Proportion of Patients', format='%')),
    color = alt.Color('Outcome:N',
                      scale=alt.Scale(
                          domain=['Pulmonary Disease', 'No Disease'],
                          range=['#DC2626', '#93C5FD']
                      ),
                      legend=alt.Legend(title='Outcome')),
    column = alt.Column('Factor:N',
                        header=alt.Header(labelFontSize=13, labelFontWeight='bold', titleFontSize=0)),
    opacity = alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=[
        alt.Tooltip('Factor:N', title='Factor'),
        alt.Tooltip('Group:N', title='Group'),
        alt.Tooltip('Outcome:N', title='Outcome'),
        alt.Tooltip('Proportion:Q', title='Proportion', format='.1%'),
        alt.Tooltip('Count:Q', title='Count'),
        alt.Tooltip('Total:Q', title='Total in group'),
    ]
).add_params(
    selection
).properties(
    width=145,
    height=280,
    title=alt.TitleParams(
        text='Disease Outcomes by Top Risk Factors',
        fontSize=16,
        anchor='middle'
    )
).configure_view(
    stroke='lightgray'
).configure_axis(
    labelFontSize=11,
    titleFontSize=12
)

alt.Chart(...)

In [4]:
#replaces the 0 and 1 with disease vs no disease
df['Disease Status'] = df['pulmonary_disease_num'].map({1: 'Pulmonary Disease', 0: 'No Disease'})
#replaces the 0 and 1 with smoker vs non-smoker
df['Smoking Status'] = df['smoking'].map({1: 'Smoker', 0: 'Non-Smoker'})

input_dropdown = alt.binding_select(
    options=[None, 'Smoker', 'Non-Smoker'],
    labels=['All', 'Smoker', 'Non-Smoker'],
    name='Smoking Status: '
)
selection = alt.selection_point(fields=['Smoking Status'], bind=input_dropdown)

chart_2 = alt.Chart(df).mark_circle(size=30, opacity=0.6).encode(
    x = alt.X('age:Q',
              scale=alt.Scale(domain=[28, 86]),
              axis=alt.Axis(title='Age (years)')),
    y = alt.Y('oxygen_saturation:Q',
              scale=alt.Scale(domain=[89, 101]),
              axis=alt.Axis(title='Oxygen Saturation (%)')),
    color = alt.Color('Disease Status:N',
                      scale=alt.Scale(
                          domain=['Pulmonary Disease', 'No Disease'],
                          range=['#DC2626', '#93C5FD']
                      ),
                      legend=alt.Legend(title='Outcome')),
    opacity = alt.condition(selection, alt.value(0.6), alt.value(0.05)),
    tooltip=[
        alt.Tooltip('age:Q', title='Age'),
        alt.Tooltip('oxygen_saturation:Q', title='Oxygen Saturation', format='.1f'),
        alt.Tooltip('Disease Status:N', title='Outcome'),
        alt.Tooltip('Smoking Status:N', title='Smoking')
    ]
).add_params(
    selection
).properties(
    width=500,
    height=350,
    title=alt.TitleParams(
        text='Age vs Oxygen Saturation by Disease Status',
        fontSize=16,
    )
).configure_view(
    strokeWidth=0
).configure_axis(
    labelFontSize=11,
    titleFontSize=12
)

chart_2.save('viz_scatter.html')